This my first week project using NVIDIA'S NEMOTRON-3-NANO-OMINI-30B-A3B MODEL...This is a reasoning model just like GPT-5 this is about calculating the entropy of a statistical data..

In [29]:
import math
from binance.client import Client
import numpy as np
import pandas as pd
from scipy.stats import entropy
import requests
import json
import os
from dotenv import load_dotenv
from openai import OpenAI

Basically this piece of code is to get the 1minute candle data from binance you don't need any API key 
for this purpose....

In [30]:
# Binance client
client = Client()

# Fetch candles
klines = client.get_klines(
    symbol='BTCUSDT',
    interval=Client.KLINE_INTERVAL_1MINUTE,
    limit=500
)

In [31]:
# DataFrame
df = pd.DataFrame(klines, columns=[
    'open_time', 'open', 'high', 'low', 'close', 'volume',
    'close_time', 'quote_asset_volume', 'num_trades',
    'taker_buy_base', 'taker_buy_quote', 'ignore'
])
print(df)

         open_time            open            high             low  \
0    1778309040000  80238.57000000  80238.58000000  80222.13000000   
1    1778309100000  80222.14000000  80222.14000000  80208.33000000   
2    1778309160000  80217.35000000  80219.65000000  80210.35000000   
3    1778309220000  80219.65000000  80281.13000000  80219.64000000   
4    1778309280000  80261.53000000  80270.00000000  80253.62000000   
..             ...             ...             ...             ...   
495  1778338740000  80332.23000000  80332.23000000  80332.22000000   
496  1778338800000  80332.23000000  80332.23000000  80321.23000000   
497  1778338860000  80321.23000000  80330.21000000  80320.00000000   
498  1778338920000  80330.21000000  80349.40000000  80330.20000000   
499  1778338980000  80349.40000000  80349.40000000  80349.39000000   

              close       volume     close_time quote_asset_volume  \
0    80222.14000000   3.32787000  1778309099999    266987.89462700   
1    80217.35000000

In [32]:
# Convert close prices
df['close'] = df['close'].astype(float)

# Compute log returns
df['returns'] = np.log(df['close'] / df['close'].shift(1))

# Remove NaN
returns = df['returns'].dropna()

# Create histogram bins
hist, bins = np.histogram(returns, bins=20)

# Convert to probabilities
probabilities = hist / hist.sum()

# Remove zero probabilities
probabilities = probabilities[probabilities > 0]

# Manual Shannon Entropy
shannon_entropy = -sum(
    p * math.log2(p)
    for p in probabilities
)

h=entropy(probabilities,base=2)

print("Shannon Entropy:", shannon_entropy)
print("Shannon Entropy using scipy:", h)

close = df["close"].astype(float)

# Log returns
returns = np.log(close / close.shift(1)).dropna()

# Variance
variance = np.var(returns)

# Gaussian entropy
G = 0.5 * math.log2(
    2 * math.pi * math.e * variance
)

print("Gaussian Entropy:", G)

Shannon Entropy: 2.752202013798435
Shannon Entropy using scipy: 2.7522020137984344
Gaussian Entropy: -10.242077636891816


since we have got the shannon entropy it is time to analyse this using the NVIDIA'S AI MODEL
here we have computed shannon entropy using scipy.stats and by using pure shannon entropy formula

we are using both gaussian entrop and shannon entropy to understand the market structure

In [33]:
load_dotenv(override=True)
api_key=os.getenv("OPENROUTER_API_KEY")
client=OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key
)

Here we will be using API calls to communicate with openrouter
first we wil send a POST requesst to the openrouter API with endpoint 

In [34]:
# First API call with reasoning
response = requests.post(
  url="https://openrouter.ai/api/v1/chat/completions",
  headers={
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json",
  },
  data=json.dumps({
    "model": "nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free",
    "messages": [
{
  "role": "system",
  "content": """
You are a quantitative market analyst.

Your task:
- Interpret Shannon entropy and Gaussian entropy
- Analyze market structure
- Detect regime conditions
- Explain whether the market is:
  - trending
  - ranging
  - volatile
  - compressing
  - chaotic

Rules:
- NEVER reveal internal reasoning
- NEVER explain your thinking process
- ONLY provide the final market interpretation
- Keep response structured and professional
"""
},
      {
  "role": "user",
  "content": f"""
Market entropy metrics:

Shannon Entropy: {shannon_entropy}

Gaussian Entropy: {G}

Interpret:
- structural randomness
- volatility randomness
- probable market regime
- trend strength
- market stability
- possible upcoming behavior

Respond like a professional quant analyst.
"""
}
      ],
    "reasoning": {"enabled": True}
  })
)

In [35]:
response = response.json()
response = response['choices'][0]['message']['content']
print(response)

**Market Interpretation**

- **Structural Randomness:** Moderate (Shannon entropy ≈ 2.75) – the market exhibits a blend of deterministic patterns and stochastic fluctuations, but not extreme chaos.  
- **Volatility Randomness:** Low (Gaussian entropy ≈ ‑10.24) – indicates compressed, low‑variance price movements, suggesting a regime of reduced volatility dispersion.  
- **Probable Market Regime:** Ranging/compressing – the combined entropy signals a stable, non‑trending environment where price action oscillates within a confined band.  
- **Trend Strength:** Weak – limited directional persistence is inferred from the entropy levels, implying any existing trend is fragile.  
- **Market Stability:** High – the negative Gaussian entropy points to a relatively stable price landscape with minimal abrupt swings.  
- **Possible Upcoming Behavior:** Expect continuation of range‑bound activity; a sustained move would require a rise in entropy (increased randomness) or a sharp expansion in volat

Entropy-Based Market Structure Analysis

This project explores the use of Shannon Entropy and Gaussian entropy as quantitative tools for analyzing financial market structure and regime behavior using live market data from Binance
. Instead of relying solely on traditional technical indicators, this approach treats the market as an information-dynamic system where randomness, volatility, and structural organization can be measured mathematically through entropy theory. The system fetches historical candle data, computes logarithmic returns, and evaluates the informational properties of price movement to understand whether the market is organized, chaotic, compressing, expanding, trending, or ranging.

Shannon entropy is used to measure structural uncertainty within the market. It evaluates how organized or random price behavior is by analyzing the probability distribution of discretized market states or return distributions. Lower Shannon entropy generally indicates increased structural organization, directional persistence, and stronger trend behavior, while higher Shannon entropy suggests randomness, noise, indecision, and range-bound conditions. In this framework, Shannon entropy acts as a measure of informational disorder and helps identify whether market movement contains exploitable structure or is dominated by stochastic behavior.

Gaussian entropy is used to measure volatility uncertainty and dispersion within continuous financial return distributions. Unlike Shannon entropy, which operates on symbolic or discretized states, Gaussian entropy is derived directly from the variance of market returns and reflects the energetic state of the market. Higher Gaussian entropy typically corresponds to volatility expansion, unstable market behavior, liquidation events, or explosive movement, whereas lower Gaussian entropy represents volatility compression, quiet accumulation phases, and reduced market energy. Since financial returns are often approximately Gaussian over short windows, Gaussian entropy provides an efficient continuous-information representation of market volatility structure.

The combination of Shannon entropy and Gaussian entropy creates a multidimensional view of market behavior. By analyzing structural randomness together with volatility randomness, the system can classify market regimes more effectively than conventional indicators. For example, low Shannon entropy combined with high Gaussian entropy may indicate a strong breakout or expansionary trend phase, while high Shannon entropy combined with low Gaussian entropy may represent inefficient low-energy chop or sideways movement. This entropy-based regime classification framework allows the system to study phase transitions in financial markets similarly to thermodynamic systems in physics, where entropy represents disorder and volatility represents market energy.

The implementation uses Python with libraries such as NumPy, pandas, requests, and math to fetch and process market data. Historical candle data is retrieved from the Binance API, converted into return series, and then transformed into probability distributions for entropy computation.

This project serves as a foundational research framework for entropy-driven quantitative finance systems. Future extensions may include rolling entropy analysis, entropy derivatives, transfer entropy, conditional entropy, spectral entropy, Monte Carlo simulations, Fourier-domain analysis, regime transition modeling, game-theoretic market dynamics, and adaptive algorithmic trading systems. The broader objective is to build an information-theoretic market intelligence engine capable of identifying hidden structural changes and evolving market states through mathematical analysis rather than conventional indicator-based heuristics.

Shannon entropy is manually implemented using the classical information-theoretic formula: H(X)=−∑i=1n​p(xi​)log2​p(xi​)
Gaussian entropy is computed using the variance of continuous return distributions:   H=1/2​log2​(2πeσ^2)
